# Sunrise and Sunset Times

This tutorial computes sunrise and sunset times for every day of 2024 at a given location, then visualizes how daylight hours change over the course of the year.

`satkit.sun.rise_set` returns the sunrise and sunset times in UTC for a given date and ground location. The results are converted to local time (US Eastern) for display. The shaded region in the plot represents daylight hours.

In [ ]:
import satkit as sk
import datetime
from zoneinfo import ZoneInfo
import matplotlib.pyplot as plt
import scienceplots  # noqa: F401

plt.style.use(["science", "no-latex"])
plt.rcParams.update({
    "mathtext.fontset": "stix",
    "font.family": "serif",
    "font.serif": ["STIX Two Text", "STIXGeneral", "DejaVu Serif"],
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 12,
    "axes.formatter.use_mathtext": True,
    "svg.fonttype": "none",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linestyle": "--",
    "axes.prop_cycle": plt.cycler(color=[
        "#0077BB", "#EE7733", "#009988", "#CC3311",
        "#33BBEE", "#EE3377", "#BBBBBB",
    ]),
})
%config InlineBackend.figure_formats = ['svg']

## Compute Rise and Set Times

For each day in 2024, compute the sunrise and sunset in UTC, then convert to US Eastern time using Python's built-in `zoneinfo` module.

In [ ]:
# Array of times for each day of 2024
basetime = sk.time(2024, 1, 1)
timearr = [basetime + sk.duration(days=i) for i in range(365)]

# Coordinates of Arlington, MA
coord = sk.itrfcoord(latitude_deg=42.1514, longitude_deg=-71.1516)


# sunrise, sunset in UTC
riseset = [sk.sun.rise_set(t, coord) for t in timearr]
rise, set = zip(*riseset)

# Convert to Eastern Time
eastern = ZoneInfo("America/New_York")
drise = [r.datetime().astimezone(eastern) for r in rise]
dset = [s.datetime().astimezone(eastern) for s in set]

# Hour of day, in [0,24]
risefrac = [r.hour + r.minute / 60 + r.second / 3600 for r in drise]
setfrac = [s.hour + s.minute / 60 + s.second / 3600 for s in dset]

# Convert hour of day to a time
risetime = [datetime.time(hour=r.hour, minute=r.minute, second=r.second) for r in drise]
settime = [datetime.time(hour=s.hour, minute=s.minute, second=s.second) for s in dset]

## Visualize

Plot sunrise and sunset times over the year.  The shaded area represents daylight hours. Note the characteristic asymmetry around the solstices and the abrupt shifts from daylight saving time transitions.

In [ ]:
def frac2str(y):
    return y.strftime("%H:%M:%S")


risestring = [frac2str(r) for r in risetime]
setstring = [frac2str(s) for s in settime]

fig, ax = plt.subplots(figsize=(8, 5))
dates = [x.as_datetime() for x in timearr]
ax.plot(dates, risefrac, color='#006450', label='Sunrise')
ax.plot(dates, setfrac, color='#006450', label='Sunset')
ax.fill_between(dates, risefrac, setfrac, alpha=0.2, color='#006450')
ax.set_xlabel("Date")
ax.set_ylabel("Local Hour of Day")
ax.set_title("Sunrise and Sunset Times for 2024 in Arlington, MA")
ax.set_ylim(0, 24)
ax.set_yticks([0, 6, 12, 18, 24])
ax.set_yticklabels(["Midnight", "6 AM", "Noon", "6 PM", "Midnight"])
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()